# A3T-GCN ablation (E0 / E1 / E2) on Kaggle

Runs `main.py` of this repository on a Kaggle GPU. **All final E0 / E1 / E2 runs (and the Stage B
selection runs) must use this same notebook and accelerator**, so that only the graph differs.

**Before running**
1. *Add data* → your private dataset with the LargeST-GBA files
   (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. *Add data* → your private dataset with the ablation artifacts
   (`E1__*.npz`, `E2__*.npz`, `sensor_filter_*.json`).
3. Settings: **Accelerator = GPU P100**, **Internet = On**.
4. Edit the *Run settings* cell, then **Save Version → Save & Run All (Commit)**.
   The run continues with the browser closed (max 12 h); results appear under *Output*.

## Run settings (edit only this cell)

In [ ]:
# Final runs: change ONLY the experiment. All hyperparameters are fixed in
# config.py and verified in section 2b.
EXPERIMENT     = "E0"      # "E0" (A3T-GCN), "E1" (+VAR), "E2" (+VAR+PEM)
RUN_NAME       = f"{EXPERIMENT}_final"

# Leave as they are for the final runs
GRAPH_FILE     = None      # None = final graph from config.py (E1/E2)
EPOCHS         = None      # None = 20 epochs from config.py
SELECTION_ONLY = False     # False = the test set is evaluated once at the end

REPO_URL       = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH    = "main"

# Data and artifacts are found automatically under /kaggle/input (section 2).



## 1. Environment: Python version, TensorFlow 2.15 (last release with the TF1 API used by the model)

In [ ]:
import sys, subprocess
print("Python", sys.version)
assert sys.version_info[:2] <= (3, 11), "TensorFlow 2.15 needs Python <= 3.11"

# Installed into the environment; main.py runs in a fresh process, so no kernel restart is needed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tensorflow[and-cuda]==2.15.1", "numpy==1.26.4", "scipy==1.13.1",
                "h5py", "pandas<3", "joblib>=1.4"], check=True)

print(subprocess.run([sys.executable, "-c",
      "import tensorflow as tf, numpy as np; print('TF', tf.__version__, '| numpy', np.__version__); "
      "print('GPUs:', tf.config.list_physical_devices('GPU'))"],
      capture_output=True, text=True).stdout)

## 2. Code (fresh clone) and data / artifacts

In [ ]:
import os, glob, shutil, subprocess, sys

WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"          # package name must be 'implementation'

# ---- 1. Fresh clone of the code ----
if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

# ---- 2. Data: the folder under /kaggle/input that contains the LargeST-GBA files ----
def find_file(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input - add the dataset (Add Input)"
    return hits[0]

DATA_DIR = os.path.dirname(find_file("gba_his_raw_2021.h5"))
print("DATA_DIR:", DATA_DIR)

os.makedirs(f"{CODE}/datasets", exist_ok=True)
DATA_TARGET = f"{CODE}/datasets/LargeST GBA"
if os.path.lexists(DATA_TARGET):
    os.remove(DATA_TARGET)
os.symlink(DATA_DIR, DATA_TARGET)
for name in ["gba_his_raw_2021.h5", "gba_rn_adj.npy", "gba_meta.csv"]:
    assert os.path.exists(f"{DATA_TARGET}/{name}"), f"missing {name} in {DATA_DIR}"

# ---- 3. Artifacts: graphs + cached sensor filter (anywhere under /kaggle/input) ----
os.makedirs(f"{CODE}/outputs/graphs", exist_ok=True)
os.makedirs(f"{CODE}/outputs/sensor_filter", exist_ok=True)
for path in glob.glob("/kaggle/input/**/*", recursive=True):
    base = os.path.basename(path)
    if base.startswith(("E1__", "E2__")) and base.endswith((".npz", ".json")):
        shutil.copy(path, f"{CODE}/outputs/graphs/")
    elif base.startswith("sensor_filter_") and base.endswith(".json"):
        shutil.copy(path, f"{CODE}/outputs/sensor_filter/")

print("graphs:", sorted(os.listdir(f"{CODE}/outputs/graphs")))
print("sensor filter cache:", os.listdir(f"{CODE}/outputs/sensor_filter"))
assert EXPERIMENT == "E0" or any(
    f.startswith(f"{EXPERIMENT}__") for f in os.listdir(f"{CODE}/outputs/graphs")
), f"{EXPERIMENT} graph files not found - add the ablation-artifacts dataset"


## 2b. Fixed hyperparameters (identical for E0, E1, E2): verified, do not edit

In [ ]:
# Fixed hyperparameters of the ablation: identical for E0, E1 and E2.
# The run stops here if the cloned code does not contain exactly these values.
EXPECTED = {
    # shared A3T-GCN / training settings
    "learning_rate": 0.001,
    "grad_clip_norm": 5.0,
    "training_epoch": 20,
    "batch_size": 8,
    "gru_units": 100,
    "seq_len": 12,
    "pre_len": 1,
    "lambda_loss": 0.0015,
    "lr_patience": 5,
    "lr_factor": 0.5,
    "train_rate": 0.8,
    "val_rate": 0.1,
    "seed": 42,
    "sensor_filter": "duplicates",
    "imputation": "gap_bands",
    # component settings (E1 / E2)
    "VAR_LAG": 3,
    "VAR_ALPHA": 1e6,
    "PEM_NU0": 1e-4,
    "PEM_NU1": 2e-3,
}

import importlib
import sys
sys.path.insert(0, WORK)
cfg_module = importlib.import_module("implementation.config.config")
cfg = cfg_module.ExperimentConfig(experiment=EXPERIMENT)

actual = {k: getattr(cfg, k) if hasattr(cfg, k) else getattr(cfg_module, k) for k in EXPECTED}
mismatch = {k: (actual[k], EXPECTED[k]) for k in EXPECTED if actual[k] != EXPECTED[k]}

for k, v in actual.items():
    print(f"{k:16s} {v}")
if EPOCHS is not None:
    print(f"NOTE: EPOCHS override = {EPOCHS} (final runs must use None)")

assert not mismatch, f"Cloned code has different hyperparameters (actual, expected): {mismatch}. Push the latest code."
cfg.validate()
print("\nFixed hyperparameters verified for", EXPERIMENT, "| graph:", cfg.graph_path)


## 3. Run

In [ ]:
cmd = [sys.executable, "-u", "-m", "implementation.main", "--experiment", EXPERIMENT]
if GRAPH_FILE:
    cmd += ["--graph", f"{CODE}/outputs/graphs/{GRAPH_FILE}"]
if EPOCHS is not None:
    cmd += ["--epochs", str(EPOCHS)]
if SELECTION_ONLY:
    cmd += ["--selection-only"]
if RUN_NAME:
    cmd += ["--run-name", RUN_NAME]
print(" ".join(cmd))

env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2", PYTHONUNBUFFERED="1")
with subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        if not any(noise in line for noise in ("WARNING:tensorflow", "Instructions for updating", "non-resource variables")):
            print(line, end="")
assert process.returncode == 0, f"main.py exited with {process.returncode}"

## 4. Results (saved as notebook output)

In [ ]:
import json
run_dir = f"{CODE}/results/{RUN_NAME or EXPERIMENT}"
results = json.load(open(f"{run_dir}/results.json"))
summary = {k: results.get(k) for k in ["experiment", "best_epoch", "best_val_rmse_original",
                                         "training_seconds", "valid_sequences", "test_metrics"]}
print(json.dumps(summary, indent=2))

out = f"{WORK}/output_{RUN_NAME or EXPERIMENT}"
shutil.copytree(run_dir, out, dirs_exist_ok=True)
print("Saved to", out, "->", os.listdir(out))

## 5. Visualization of this run

Training curve for every run; for final runs also the test metrics, MAE by time of day, per-sensor MAE and observed vs predicted. Figures are saved into the output folder. The E0 / E1 / E2 comparison is produced locally with `compare_results.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
try:
    from IPython.display import display
except ImportError:          # plain Python (outside a notebook)
    display = print

BLUE, INK, MUTED, GRID, SURFACE = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
})

label = RUN_NAME or EXPERIMENT
history = pd.read_csv(f"{out}/history.csv") if os.path.exists(f"{out}/history.csv") \
    else pd.DataFrame(results["history"])

# 1. Training curve: validation RMSE / MAE (top), learning rate (bottom, own axis)
fig, (ax, ax_lr) = plt.subplots(2, 1, figsize=(7.5, 5), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})
ax.plot(history["epoch"], history["val_rmse_original"], color=BLUE, marker="o", ms=4, label="Val RMSE")
ax.plot(history["epoch"], history["val_mae_original"], color=BLUE, ls="--", marker="o", ms=3, label="Val MAE")
best = history["val_rmse_original"].idxmin()
ax.plot(history["epoch"][best], history["val_rmse_original"][best], "o", ms=10,
        mfc=SURFACE, mec=BLUE, mew=2, label=f"best epoch {int(history['epoch'][best])}")
ax.set_ylabel("Vehicles / 5 min")
ax.set_title(f"{label}: validation error per epoch")
ax.legend()
ax_lr.step(history["epoch"], history["learning_rate"], where="post", color=MUTED)
ax_lr.set_ylabel("LR")
ax_lr.set_xlabel("Epoch")
ax_lr.xaxis.set_major_locator(MaxNLocator(integer=True))
fig.tight_layout(); fig.savefig(f"{out}/fig_training_curve.png", dpi=200); plt.show()

if not SELECTION_ONLY and os.path.exists(f"{out}/test_predictions.npz"):

    # 2. Test metrics table
    metrics = {k: v for k, v in results["test_metrics"].items() if k != "mape_min_flow"}
    table = pd.DataFrame([metrics], index=[label])
    table["best_epoch"] = results["best_epoch"]
    table["best_val_rmse"] = results["best_val_rmse_original"]
    table["training_h"] = results["training_seconds"] / 3600
    display(table.round(4))
    display(pd.DataFrame(results["test_breakdown"]).T.round(4))

    pred = np.load(f"{out}/test_predictions.npz")
    per_sensor = pd.read_csv(f"{out}/per_sensor_metrics.csv")
    tod = pd.read_csv(f"{out}/time_of_day_metrics.csv")

    # 3. MAE by time of day and per-sensor MAE distribution
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
    a1.plot(tod["slot"] * 5 / 60, tod["mae"].rolling(3, center=True, min_periods=1).mean(), color=BLUE)
    a1.set_xlim(0, 24); a1.set_xticks(range(0, 25, 3))
    a1.set_xlabel("Time of day (h)"); a1.set_ylabel("Test MAE (veh / 5 min)")
    a1.set_title("Test MAE by time of day")
    a2.hist(per_sensor["mae"], bins=50, color=BLUE, edgecolor=SURFACE, linewidth=0.5)
    a2.axvline(per_sensor["mae"].median(), color=INK, ls="--", lw=1,
               label=f"median {per_sensor['mae'].median():.1f}")
    a2.set_xlabel("Per-sensor test MAE"); a2.set_ylabel("Sensors")
    a2.set_title(f"Per-sensor MAE ({len(per_sensor)} sensors)"); a2.legend()
    fig.tight_layout(); fig.savefig(f"{out}/fig_test_errors.png", dpi=200); plt.show()

    # 4. Observed vs predicted, first 3 test days, median- and high-flow sensor
    times = pd.to_datetime(pred["timestamps_ns"])
    window = times < times[0] + pd.Timedelta(days=3)
    order = np.argsort(pred["targets"].mean(axis=0))
    picks = [order[len(order) // 2], order[int(0.9 * (len(order) - 1))]]
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    for a, j in zip(axes, picks):
        a.plot(times[window], pred["targets"][window, j], color=INK, lw=1.1, label="Observed")
        a.plot(times[window], pred["predictions"][window, j], color=BLUE, lw=1.0, label=label)
        a.set_title(f"Sensor {int(pred['sensor_ids'][j])}"); a.set_ylabel("Flow"); a.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(f"{out}/fig_example_series.png", dpi=200); plt.show()

print("Figures saved in", out, "->", [f for f in os.listdir(out) if f.startswith("fig_")])
